In [1]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

In [2]:
import re
from tqdm import tqdm
import glob

In [3]:
# define the question logs path dictionary 
log_path_dict = {
    'qald9plus_test': ['data_dir/processed_kgqa_ds/qald9plus/test/ablation.test.prediction/tentrisq10_aug_gold/logs/en__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b'],
    'qald10_test': ['data_dir/processed_kgqa_ds/qald10/test/ablation.test.prediction/tentrisq10_aug_gold/logs/en__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b'],
    'lcquad2_test': ['data_dir/processed_kgqa_ds/lcquad2/test/test.prediction/tentrisq10_aug_gold/logs/en__lola__PBSG_MHOP__t20-h-1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b']
}

In [4]:
# For each dataset
    # For each logs directory
        # For each *.txt in the directory
            # Fetch the content
            # count pattern extraction queries
                # count startswith "Triple patterns found for" # these many queries send to retrieve patterns
                # looks for "Selected top (%d+) .+" # add %d to existing edge count
                # total edge count is the total number of queries sent to extract concrete instances
# print dataset specific average number of queries and an overall macro and micro average as well

In [5]:
# Helper regexes
PATTERN_EXTRACT_RE = re.compile(r"Triple patterns found for", re.MULTILINE)
SELECTED_TOP_RE   = re.compile(r"Selected top\s+(\d+)", re.MULTILINE)

def _extract_counts_from_file(file_path: str) -> tuple[int, int]:
    """
    Reads a log file and returns a tuple:
        (num_pattern_queries, total_selected_top)

    * ``num_pattern_queries`` – how many times a line starts with
      ``Triple patterns found for`` (each such line represents one SPARQL
      request for pattern extraction).

    * ``total_selected_top`` – sum of all integer values captured by
      ``Selected top <N>`` (the number of concrete‑instance queries sent).
    """
    with open(file_path, "r", encoding="utf-8") as f:
        txt = f.read()

    # Count pattern‑extraction queries
    pattern_queries = len(PATTERN_EXTRACT_RE.findall(txt))

    # Sum up all “Selected top N …” occurrences
    selected_top_sum = sum(int(m) for m in SELECTED_TOP_RE.findall(txt))

    return pattern_queries, selected_top_sum


def analyse_dataset(dataset_name: str, log_dirs: list) -> dict:
    """
    Walks through each ``*.txt`` log file in the supplied directories,
    extracts the two metrics defined above and returns a dict:

        {
            "files":      <number of log files examined>,
            "patterns":   <total pattern‑extraction queries>,
            "edges":      <total “selected top” count>,
            "avg_patterns": <patterns / files>,
            "avg_edges":    <edges / files>,
        }
    """
    total_files = 0
    total_patterns = 0
    total_edges = 0

    for rel_dir in log_dirs:
        abs_dir = get_fp(rel_dir)
        # Recursively grab every *.txt file
        pattern = os.path.join(abs_dir, "**", "*.txt")
        for file_path in tqdm(glob.glob(pattern, recursive=True),
                              desc=f"Scanning {dataset_name}",
                              unit="file"):
            total_files += 1
            p_cnt, e_cnt = _extract_counts_from_file(file_path)
            total_patterns += p_cnt
            total_edges += e_cnt

    avg_patterns = (total_patterns / total_files) if total_files else 0.0
    avg_edges    = (total_edges    / total_files) if total_files else 0.0

    return {
        "files": total_files,
        "patterns": total_patterns,
        "edges": total_edges,
        "avg_patterns": avg_patterns,
        "avg_edges": avg_edges,
    }

In [6]:
# Main aggregation over all datasets
dataset_stats = {}
overall_files = overall_patterns = overall_edges = 0
macro_avg_patterns_sum = macro_avg_edges_sum = 0.0

for ds_name, dirs in log_path_dict.items():
    stats = analyse_dataset(ds_name, dirs)
    dataset_stats[ds_name] = stats

    overall_files     += stats["files"]
    overall_patterns  += stats["patterns"]
    overall_edges     += stats["edges"]
    macro_avg_patterns_sum += stats["avg_patterns"]
    macro_avg_edges_sum    += stats["avg_edges"]

# Micro‑averages (global totals)
micro_avg_patterns = (overall_patterns / overall_files) if overall_files else 0.0
micro_avg_edges    = (overall_edges    / overall_files) if overall_files else 0.0

# Macro‑averages (mean of per‑dataset averages)
num_datasets = len(dataset_stats)
macro_avg_patterns = (macro_avg_patterns_sum / num_datasets) if num_datasets else 0.0
macro_avg_edges    = (macro_avg_edges_sum    / num_datasets) if num_datasets else 0.0

# ----------------------------------------------------------------------
# Pretty‑print the results
print("\n=== SPARQL Statistics ===\n")
for ds, stats in dataset_stats.items():
    print(f"{ds:>12}: {stats['files']} files | "
          f"pattern retrievals: {stats['patterns']} ({stats['avg_patterns']:.2f} per file) | "
          f"edges: {stats['edges']} ({stats['avg_edges']:.2f} per file)")

print("\nOverall (micro) averages:")
print(f"  Pattern retrievals per file: {micro_avg_patterns:.2f}")
print(f"  Edges per file   : {micro_avg_edges:.2f}")

print("\nOverall (macro) averages:")
print(f"  Pattern retrievals per file: {macro_avg_patterns:.2f}")
print(f"  Edges per file   : {macro_avg_edges:.2f}\n")

Scanning lcquad2_test: 100%|██████████| 3997/3997 [00:00<00:00, 19103.17file/s]


=== SPARQL Statistics ===

qald9plus_test: 123 files | pattern retrievals: 187 (1.52 per file) | edges: 2258 (18.36 per file)
 qald10_test: 393 files | pattern retrievals: 911 (2.32 per file) | edges: 11641 (29.62 per file)
lcquad2_test: 3997 files | pattern retrievals: 6955 (1.74 per file) | edges: 93181 (23.31 per file)

Overall (micro) averages:
  Pattern retrievals per file: 1.78
  Edges per file   : 23.73

Overall (macro) averages:
  Pattern retrievals per file: 1.86
  Edges per file   : 23.76

